In [ ]:
!pip install lightgbm catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
import lightgbm as lgb
import catboost as cb

from google.colab import drive
drive.mount('/content/drive')

# 1. Load Dataset
file_path = "/content/drive/MyDrive/MLOps project/cleaned_dataset.csv"
df = pd.read_csv(file_path)

# 2. DATA CLEANING & PREPROCESSING (Sir's requested drops)
cols_to_remove = [
    'user_rating', 'num_reviews', 'digital_display',
    'url', 'variant', 'charging_time_hrs', 'kerb_weight_kg'
]
df = df.drop(columns=cols_to_remove, errors='ignore')

# Impute missing values with Median to keep all 364 rows
num_cols = ['ex_showroom_price_inr', 'battery_capacity_kwh', 'range_km', 'top_speed_kmph', 'motor_power']
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Convert Boolean to 1/0
bool_cols = ['bluetooth', 'navigation', 'riding_modes', 'fast_charging']
for col in bool_cols:
    df[col] = df[col].astype(int)

# Encode Categorical Text to Numbers
le = LabelEncoder()
cat_cols = ['brand', 'segment', 'brand_country']
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

# Remove duplicates
df = df.drop_duplicates()

# 3. DEFINE TARGET AND FEATURES
target_column_name = 'ex_showroom_price_inr'

# We drop 'scooter_name' because it's a unique label, not a predictive feature
X = df.drop(columns=[target_column_name, 'scooter_name'], errors='ignore')
y = df[target_column_name]

print(f"✅ Preprocessing complete. Target: {target_column_name}")
print(f"✅ Features being used: {X.columns.tolist()}")

# Split into train/test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. DEFINE MODELS
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Lasso Regression": Lasso(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Support Vector Regressor": SVR(),
    "XGBoost Regressor": XGBRegressor(objective='reg:squarederror', random_state=42),
    "LightGBM Regressor": lgb.LGBMRegressor(random_state=42, verbose=-1),
    "CatBoost Regressor": cb.CatBoostRegressor(random_state=42, verbose=0)
}

# Store results
results = []
feature_columns = X.columns.tolist()

# 5. EVALUATE EACH MODEL
for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    end = time.time()

    # Calculate MAPE
    non_zero_indices = y_test != 0
    mape = np.mean(np.abs((y_test[non_zero_indices] - y_pred[non_zero_indices]) / y_test[non_zero_indices])) * 100

    results.append({
        "Model": name,
        "R² Score": round(r2_score(y_test, y_pred), 4),
        "MAE": round(mean_absolute_error(y_test, y_pred), 4),
        "MAPE (%)": round(mape, 2),
        "Train Time (s)": round(end - start, 3)
    })

    print(f"\n--- Model Explainability for {name} ---")
    try:
        if hasattr(model, 'coef_'):
            importances = pd.Series(model.coef_, index=feature_columns)
            print(importances.sort_values(ascending=False).to_string())
        elif hasattr(model, 'feature_importances_'):
            importances = pd.Series(model.feature_importances_, index=feature_columns)
            print(importances.sort_values(ascending=False).to_string())
        else:
            print("No direct feature importance available (Common for SVR).")
    except Exception as e:
        print(f"Error extracting importance: {e}")

# 6. DISPLAY SUMMARY
results_df = pd.DataFrame(results).sort_values(by="MAE")
print("\n🔍 Model Evaluation Summary:")
print(results_df.to_string(index=False))

best_model_name = results_df.iloc[0]['Model']
print(f"\n🏆 The best performing model is: {best_model_name}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Preprocessing complete. Target: ex_showroom_price_inr
✅ Features being used: ['brand', 'battery_capacity_kwh', 'range_km', 'top_speed_kmph', 'motor_power', 'bluetooth', 'navigation', 'riding_modes', 'fast_charging', 'segment', 'brand_value', 'brand_country', 'brand_model_count', 'price_per_km', 'performance_score', 'brand_encoded']

--- Model Explainability for Linear Regression ---
segment                 13381.445026
brand_country            8216.631652
battery_capacity_kwh     5071.757331
brand_model_count        1748.695343
brand_value              1718.685604
riding_modes              601.734021
range_km                  388.563198
bluetooth                 280.106124
performance_score         160.891823
price_per_km               78.024509
top_speed_kmph             30.843032
brand_encoded             -15.404497
brand                     -15.404497
mo

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.920e+08, tolerance: 2.627e+07
  model = cd_fast.enet_coordinate_descent(



--- Model Explainability for Random Forest ---
segment                 0.720193
price_per_km            0.102465
range_km                0.062616
battery_capacity_kwh    0.033944
performance_score       0.025261
top_speed_kmph          0.023038
brand                   0.010305
brand_encoded           0.006658
motor_power             0.005612
bluetooth               0.002537
navigation              0.002247
brand_model_count       0.001895
riding_modes            0.001054
fast_charging           0.000948
brand_value             0.000699
brand_country           0.000526

--- Model Explainability for Support Vector Regressor ---
No direct feature importance available (Common for SVR).

--- Model Explainability for XGBoost Regressor ---
segment                 9.576660e-01
range_km                1.916083e-02
price_per_km            8.426731e-03
top_speed_kmph          5.917802e-03
performance_score       3.207801e-03
battery_capacity_kwh    2.044759e-03
motor_power             1.864066e-

In [ ]:
# --- 1. REMOVE LEAKY FEATURES ---
# We keep only the physical specs and brand info
leaky_cols = ['price_per_km', 'performance_score', 'brand_encoded']
X_honest = X.drop(columns=leaky_cols, errors='ignore')

print(f"✅ Leaky features removed. training on: {X_honest.columns.tolist()}")

# --- 2. RE-RUN BEST MODELS WITH CROSS-VALIDATION ---
from sklearn.model_selection import cross_val_score

# We'll use your top 3 winners from last time
honest_models = {
    "CatBoost": cb.CatBoostRegressor(random_state=42, verbose=0),
    "Random Forest": RandomForestRegressor(random_state=42),
    "XGBoost": XGBRegressor(objective='reg:squarederror', random_state=42)
}

print("\n📊 'Honest' Cross-Validation Results (R² Score):")
for name, model in honest_models.items():
    # cv=5 means it tests the model 5 different times on different data slices
    scores = cross_val_score(model, X_honest, y, cv=5, scoring='r2')
    print(f"{name}: {scores.mean():.4f} (+/- {scores.std() * 2:.2f})")

✅ Leaky features removed. training on: ['brand', 'battery_capacity_kwh', 'range_km', 'top_speed_kmph', 'motor_power', 'bluetooth', 'navigation', 'riding_modes', 'fast_charging', 'segment', 'brand_value', 'brand_country', 'brand_model_count']

📊 'Honest' Cross-Validation Results (R² Score):
CatBoost: 0.7504 (+/- 0.13)
Random Forest: 0.7600 (+/- 0.11)
XGBoost: 0.7290 (+/- 0.20)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error

# 1. LOAD AND CLEAN (Same as before to ensure no errors)
df = pd.read_csv('/content/drive/MyDrive/MLOps project/cleaned_dataset.csv')

# Remove Sir's noisy columns + Redundant + Leaky features
cols_to_drop = [
    'user_rating', 'num_reviews', 'digital_display', 'url', 'variant',
    'charging_time_hrs', 'kerb_weight_kg', 'scooter_name',
    'price_per_km', 'performance_score', 'brand_encoded' # LEAKAGE REMOVAL
]
df_ml = df.drop(columns=cols_to_drop, errors='ignore')

# Impute and Encode
for col in df_ml.select_dtypes(include=[np.number]).columns:
    df_ml[col] = df_ml[col].fillna(df_ml[col].median())

le = LabelEncoder()
for col in ['brand', 'segment', 'brand_country']:
    df_ml[col] = le.fit_transform(df_ml[col].astype(str))

for col in df_ml.select_dtypes(include=['bool']).columns:
    df_ml[col] = df_ml[col].astype(int)

# 2. DEFINE X AND Y
X = df_ml.drop(columns=['ex_showroom_price_inr'])
y = df_ml['ex_showroom_price_inr']

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. HYPERPARAMETER TUNING (Finding the Optimal Balance)
print("Searching for optimal parameters... this may take a minute.")
param_grid = {
    'n_estimators': [100, 200, 500],        # Number of trees
    'max_depth': [10, 20, 30, None],       # Control Overfitting (Pruning)
    'min_samples_split': [2, 5, 10],       # Higher = Less Overfitting
    'max_features': ['sqrt', 'log2', None] # Diversity of trees
}

grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1 # Uses all processor cores
)

grid_search.fit(X_train_h, y_train_h)

# 4. RESULTS
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_h)

print(f"\n✅ TUNING COMPLETE")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"New 'Honest' R² Score: {r2_score(y_test_h, y_pred):.4f}")
print(f"New MAE: ₹{mean_absolute_error(y_test_h, y_pred):.2f}")

Searching for optimal parameters... this may take a minute.

✅ TUNING COMPLETE
Best Parameters: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 100}
New 'Honest' R² Score: 0.8748
New MAE: ₹10432.97


In [1]:
html_form = """
<!DOCTYPE html>
<html>
<head>
    <title>EV Scooter Price Predictor</title>
    <style>
        body { font-family: sans-serif; margin: 40px; background-color: #f4f4f4; }
        .container { background: white; padding: 20px; border-radius: 10px; max-width: 500px; margin: auto; }
        input, select { width: 100%; padding: 10px; margin: 10px 0; border: 1px solid #ddd; }
        button { background: #28a745; color: white; padding: 10px; border: none; width: 100%; cursor: pointer; }
    </style>
</head>
<body>
    <div class="container">
        <h2>Scooter Price Predictor</h2>
        <form id="predictForm">
            <label>Battery Capacity (kWh)</label>
            <input type="number" step="0.1" name="battery_capacity_kwh" value="3.0" required>

            <label>Range (km)</label>
            <input type="number" name="range_km" value="100" required>

            <label>Top Speed (kmph)</label>
            <input type="number" name="top_speed_kmph" value="80" required>

            <label>Motor Power (W)</label>
            <input type="number" name="motor_power" value="3000" required>

            <label>Segment (0:Commuter, 1:Performance, 2:Premium)</label>
            <select name="segment">
                <option value="0">Commuter</option>
                <option value="1">Performance</option>
                <option value="2">Premium</option>
            </select>

            <input type="hidden" name="brand" value="1">
            <input type="hidden" name="brand_value" value="5">
            <input type="hidden" name="brand_country" value="1">
            <input type="hidden" name="brand_model_count" value="3">
            <input type="hidden" name="bluetooth" value="1">
            <input type="hidden" name="navigation" value="1">
            <input type="hidden" name="riding_modes" value="1">
            <input type="hidden" name="fast_charging" value="1">

            <button type="submit">Predict Launch Price</button>
        </form>
        <h3 id="result" style="text-align:center; color:#28a745;"></h3>
    </div>

    <script>
        document.getElementById('predictForm').addEventListener('submit', async (e) => {
            e.preventDefault();
            const formData = new FormData(e.target);
            const data = Object.fromEntries(formData.entries());

            const response = await fetch('/predict', {
                method: 'POST',
                headers: {'Content-Type': 'application/json'},
                body: JSON.stringify(data)
            });
            const result = await response.json();
            document.getElementById('result').innerText = 'Predicted Price: ₹' + result.prediction.toLocaleString();
        });
    </script>
</body>
</html>
"""

In [3]:
!pip install pyngrok flask

In [ ]:
import os
from flask import Flask, request, jsonify, render_template_string
import pandas as pd
import numpy as np
from pyngrok import ngrok

# Replace the text inside the quotes with your actual token from Step 1
ngrok.set_auth_token("34mOxSGipoucomQGaDZkIOrDFR4_R4YpEXgRezXinfWM1w3H")
# 1. Start Ngrok Tunnel
ngrok.kill()
public_url = ngrok.connect(5000)
print(f"✅ Website is LIVE at: {public_url}")

app = Flask(__name__)
import joblib

# 1. Load the actual file into a variable
try:
    model_to_use = joblib.load('model.pkl')
    print("✅ Model loaded successfully from file!")
except Exception as e:
    print(f"❌ Error loading model file: {e}")

# 2. Make sure 'X' (your training data) is defined so we can get column names
# If 'X' isn't available, manually define the columns:
feature_columns = [
    'brand', 'battery_capacity_kwh', 'range_km', 'top_speed_kmph',
    'motor_power', 'bluetooth', 'navigation', 'riding_modes',
    'fast_charging', 'segment', 'brand_value', 'brand_country',
    'brand_model_count'
]


html_form = """
<!DOCTYPE html>
<html>
<head>
    <title>EV Scooter Launch Price Predictor</title>
    <style>
        body { font-family: sans-serif; margin: 30px; background-color: #f0f2f5; }
        .card { background: white; padding: 25px; border-radius: 12px; max-width: 500px; margin: auto; box-shadow: 0 4px 10px rgba(0,0,0,0.1); }
        h2 { color: #1a73e8; text-align: center; }
        label { display: block; margin-top: 10px; font-weight: bold; font-size: 0.9em; }
        input, select { width: 100%; padding: 10px; margin-top: 5px; border: 1px solid #ddd; border-radius: 6px; box-sizing: border-box; }
        button { background: #1a73e8; color: white; border: none; padding: 12px; width: 100%; border-radius: 6px; margin-top: 20px; cursor: pointer; font-size: 1em; }
        #result { margin-top: 20px; text-align: center; font-size: 1.3em; font-weight: bold; color: #2e7d32; }
    </style>
</head>
<body>
    <div class="card">
        <h2>Scooter Price Estimator</h2>
        <form id="priceForm">
            <label>Battery Capacity (kWh)</label>
            <input type="number" step="0.1" name="battery_capacity_kwh" value="2.9" required>

            <label>Range (km)</label>
            <input type="number" name="range_km" value="115" required>

            <label>Top Speed (kmph)</label>
            <input type="number" name="top_speed_kmph" value="80" required>

            <label>Motor Power (W)</label>
            <input type="number" name="motor_power" value="3000" required>

            <label>Segment</label>
            <select name="segment">
                <option value="0">Budget/Commuter</option>
                <option value="1">Performance</option>
                <option value="2">Premium</option>
            </select>

            <label>Brand Reputation (1-10)</label>
            <input type="number" name="brand_value" value="7" required>

            <input type="hidden" name="brand" value="1">
            <input type="hidden" name="brand_country" value="1">
            <input type="hidden" name="brand_model_count" value="5">
            <input type="hidden" name="bluetooth" value="1">
            <input type="hidden" name="navigation" value="1">
            <input type="hidden" name="riding_modes" value="1">
            <input type="hidden" name="fast_charging" value="1">

            <button type="submit">Predict Price</button>
        </form>
        <div id="result"></div>
    </div>

    <script>
        document.getElementById('priceForm').addEventListener('submit', async (e) => {
            e.preventDefault();
            const formData = Object.fromEntries(new FormData(e.target));
            const response = await fetch('/predict', {
                method: 'POST',
                headers: {'Content-Type': 'application/json'},
                body: JSON.stringify(formData)
            });
            const data = await response.json();
            document.getElementById('result').innerText = 'Predicted Price: ₹' + Math.round(data.prediction).toLocaleString();
        });
    </script>
</body>
</html>
"""

@app.route("/")
def index():
    return render_template_string(html_form)

@app.route("/predict", methods=["POST"])
def predict():
    try:
        data = request.get_json()
        input_df = pd.DataFrame([data])
        # Ensure correct column order
        input_df = input_df[feature_columns].apply(pd.to_numeric)
        prediction = model_to_use.predict(input_df)
        return jsonify({'prediction': float(prediction[0])})
    except Exception as e:
        return jsonify({'error': str(e)}), 400

print("Starting Flask app...")
app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

✅ Website is LIVE at: NgrokTunnel: "https://ditriglyphic-ok-indiscriminately.ngrok-free.dev" -> "http://localhost:5000"
✅ Model loaded successfully from file!
Starting Flask app...
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [23/Apr/2026 10:27:58] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/Apr/2026 10:27:59] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [23/Apr/2026 10:28:08] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/Apr/2026 10:29:13] "POST /predict HTTP/1.1" 200 -
